In [ ]:
!pip install timm==1.0.3 albumentations -q
print('OK')

In [ ]:
import gc, random, time, warnings
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
warnings.filterwarnings('ignore')
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
DEBUG        = True
DEBUG_EPOCHS = 2
EPOCHS       = DEBUG_EPOCHS if DEBUG else 79

KAGGLE_DATA = None
for candidate in [
    Path('/kaggle/input/m2-food-calorie-estimation'),
    Path('/kaggle/input/competitions/m2-food-calorie-estimation'),
]:
    if candidate.exists():
        KAGGLE_DATA = candidate; break
if KAGGLE_DATA is None:
    KAGGLE_DATA = list(Path('/kaggle/input').rglob('train_labels.csv'))[0].parent

WORK     = Path('/kaggle/working')
CKPT_DIR = WORK / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)

SEED=42; PRED_MIN=30.0; PRED_MAX=3600.0; NUM_WORKERS=2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, '|', KAGGLE_DATA)

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True

TRAIN_IMGS = KAGGLE_DATA / 'train' / 'images'
TEST_IMGS  = KAGGLE_DATA / 'test'  / 'images'

def resolve_path(img_dir, filename):
    p = img_dir / filename
    if p.exists(): return str(p)
    alt = p.with_suffix('.png' if p.suffix.lower()=='.jpg' else '.jpg')
    return str(alt) if alt.exists() else str(p)

train_df = pd.read_csv(KAGGLE_DATA / 'train_labels.csv')
test_df  = pd.read_csv(KAGGLE_DATA / 'test_ids.csv')
train_df['path'] = train_df['filename'].apply(lambda f: resolve_path(TRAIN_IMGS, f))
test_df['path']  = test_df['filename'].apply(lambda f: resolve_path(TEST_IMGS, f))
train_df['target_log'] = np.log1p(train_df['calories'].astype(float))
print(f'train={len(train_df)} test={len(test_df)}')

In [ ]:
# ConvNextV2-Base image-only — pas d'embeddings texte
# Architecture identique au modèle multimodal mais sans la branche texte
BACKBONE = 'convnextv2_base.fcmae_ft_in22k_in1k'
IMG_SIZE = 224; BS = 32
LR=2e-4; WD=1e-2; DROPOUT=0.25
MIXUP_ALPHA=0.4; AUG_MIX_PROB=0.3

class ImageDataset(Dataset):
    def __init__(self, df, transforms, is_train=True):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.is_train = is_train
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.cvtColor(cv2.imread(row.path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        img = self.transforms(image=img)['image']
        if self.is_train:
            return img, torch.tensor(float(row.target_log), dtype=torch.float32)
        return img, str(row.image_id)

class ImageOnlyRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0, global_pool='avg')
        d = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, 512), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(512, 128), nn.GELU(), nn.Dropout(DROPOUT/2),
            nn.Linear(128, 1),
        )
        print(f'features={d}')
    def forward(self, img):
        return self.head(self.backbone(img)).squeeze(1)

print('Classes définies OK')

In [ ]:
# ------------------------------------------------------------------------
# DATA AUGMENTATION AGRESSIVE — Pipeline d'entraînement
#
# Ce modèle utilise une augmentation plus agressive que le modèle MM
# pour compenser l'absence d'information textuelle et forcer le réseau
# à apprendre des features visuelles plus robustes.
#
# 1. RandomResizedCrop (scale=0.5-1.0) — plus agressif (50% min vs 65%)
#    Force le modèle à reconnaître les plats même très recadrés.
#
# 2. HorizontalFlip (p=0.5) + VerticalFlip (p=0.2) + RandomRotate90 (p=0.3)
#    Invariance à l'orientation dans toutes les directions.
#
# 3. ColorJitter très fort (brightness=0.4, contrast=0.4, saturation=0.4)
#    Variations de couleur plus importantes pour la robustesse.
#
# 4. GaussianBlur (p=0.3) — simule des photos floues ou de mauvaise qualité.
#
# 5. GridDistortion (p=0.2) — déformations géométriques légères.
#
# 6. CoarseDropout (p=0.3) — masque aléatoirement des zones de l'image.
#    Force le modèle à ne pas se fier à une seule partie du plat.
#
# ------------------------------------------------------------------------

train_tfm = A.Compose([
    A.RandomResizedCrop(size=(IMG_SIZE,IMG_SIZE), scale=(0.5,1.0), ratio=(0.75,1.33), interpolation=cv2.INTER_AREA),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.2), A.RandomRotate90(p=0.3),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.GaussianBlur(blur_limit=(3,7), p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, fill_value=0, p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
])
test_tfm = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE, border_mode=cv2.BORDER_REFLECT_101),
    A.CenterCrop(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2(),
])

train_loader = DataLoader(ImageDataset(train_df, train_tfm, True),
                          batch_size=BS, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
test_loader  = DataLoader(ImageDataset(test_df,  test_tfm, False),
                          batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'train batches={len(train_loader)} test batches={len(test_loader)}')

In [ ]:
model     = ImageOnlyRegressor().to(device)
criterion = nn.HuberLoss(delta=0.5)
scaler    = GradScaler(enabled=torch.cuda.is_available())

bp = [(n,p) for n,p in model.backbone.named_parameters() if p.requires_grad]
hp = [(n,p) for n,p in model.head.named_parameters()    if p.requires_grad]
gs = max(1, len(bp)//4); pg = []
for g in range(4):
    chunk = bp[g*gs:(g+1)*gs if g<3 else len(bp)]
    lr_g  = LR * (0.25**(3-g))
    wd_  = [p for n,p in chunk if p.ndim>1 and 'bias' not in n and 'norm' not in n.lower()]
    nwd  = [p for n,p in chunk if not (p.ndim>1 and 'bias' not in n and 'norm' not in n.lower())]
    if wd_:  pg.append({'params':wd_,  'lr':lr_g, 'weight_decay':WD})
    if nwd:  pg.append({'params':nwd,  'lr':lr_g, 'weight_decay':0.})
hwd  = [p for n,p in hp if p.ndim>1 and 'bias' not in n and 'norm' not in n.lower()]
hnwd = [p for n,p in hp if not (p.ndim>1 and 'bias' not in n and 'norm' not in n.lower())]
if hwd:  pg.append({'params':hwd,  'lr':LR, 'weight_decay':WD})
if hnwd: pg.append({'params':hnwd, 'lr':LR, 'weight_decay':0.})

optimizer = torch.optim.AdamW(pg)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS*len(train_loader), eta_min=1e-7)
print('Optimizer OK')

In [ ]:
ckpt_path = CKPT_DIR / 'convnextv2_base_augmax.pth'
best_loss = float('inf')

for epoch in range(1, EPOCHS+1):
    t0 = time.time(); model.train(); total = 0.
    optimizer.zero_grad(set_to_none=True)
    for imgs, targets in train_loader:
        imgs    = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if random.random() < AUG_MIX_PROB:
            lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
            idx = torch.randperm(imgs.size(0), device=device)
            imgs = lam*imgs+(1-lam)*imgs[idx]
            targets = lam*targets+(1-lam)*targets[idx]
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            loss = criterion(model(imgs), targets)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        if scheduler: scheduler.step()
        total += float(loss.item())
    train_loss = total / len(train_loader)
    if train_loss < best_loss:
        best_loss = train_loss
        torch.save({'model_state': model.state_dict(), 'epoch': epoch}, ckpt_path)
    print(f'  {epoch:03d}/{EPOCHS} | train={train_loss:.4f} best={best_loss:.4f} | {time.time()-t0:.0f}s', flush=True)

In [ ]:
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()

all_ids, all_preds = [], []
with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast(device_type='cuda', enabled=torch.cuda.is_available()):
            out      = model(imgs)
            out_flip = model(torch.flip(imgs, dims=[3]))
        all_preds.append(((out+out_flip)/2).float().cpu().numpy())
        all_ids.extend(list(ids))

preds    = np.concatenate(all_preds)
calories = np.clip(np.expm1(preds), PRED_MIN, PRED_MAX)
sub = pd.DataFrame({'image_id': all_ids, 'calories': calories.round(2)})
sub.to_csv(WORK / 'submission_convnextv2_base_augmax.csv', index=False)
print(f'mean={calories.mean():.0f} min={calories.min():.0f} max={calories.max():.0f}')
print(sub.head())